# FASE 4: Modelos Generativos

**Autor:** Jaime Meléndez Zambrano

## Objetivo
Construir generadores de texto usando **cadenas de Markov** (orden 1 y
orden 2), uno por cada categoría de sentimiento, y comparar la calidad y
coherencia del texto generado entre órdenes. Con el corpus ampliado (12
encuestadores) cada cadena tiene muchos más estados que en la versión
anterior, lo que debería notarse en generaciones menos repetitivas.

In [1]:
import os
import sys
import random
import pandas as pd

sys.path.insert(0, os.path.abspath('../src'))
from procesador_nlp import ProcesadorNLP
from generador_markov import GeneradorMarkov

random.seed(42)


In [2]:
procesador = ProcesadorNLP()
textos, etiquetas = procesador.cargar_corpus('../data/corpus_etiquetado')

textos_por_categoria = {
    cat: [t for t, e in zip(textos, etiquetas) if e == cat]
    for cat in sorted(set(etiquetas))
}
for cat, lista in textos_por_categoria.items():
    print(f"{cat}: {len(lista)} textos")


Corpus cargado: 843 textos
Categorías: ['desagradable', 'mixto', 'placentero']
desagradable: 325 textos
mixto: 205 textos
placentero: 313 textos


## Entrenar los modelos (orden 1 y orden 2) para cada categoría

In [3]:
modelos_markov = {}
for categoria, lista_textos in textos_por_categoria.items():
    m1 = GeneradorMarkov(orden=1)
    m2 = GeneradorMarkov(orden=2)
    m1.entrenar_multiples(lista_textos)
    m2.entrenar_multiples(lista_textos)
    modelos_markov[categoria] = {"orden1": m1, "orden2": m2}
    print()


Entrenado con 325 textos (orden 1)
Estados distintos: 768
Entrenado con 325 textos (orden 2)
Estados distintos: 2159

Entrenado con 205 textos (orden 1)
Estados distintos: 799
Entrenado con 205 textos (orden 2)
Estados distintos: 2117

Entrenado con 313 textos (orden 1)
Estados distintos: 749
Entrenado con 313 textos (orden 2)
Estados distintos: 2145



## Generación de textos

**Nota de adaptación:** la guía original sugiere prompts de estilo
literario ("Había una vez...", "El principio de todo..."), pensados para
cuentos y fábulas. Como nuestro corpus son respuestas de encuesta en
primera persona, se usan prompts adaptados a ese registro: `"Siento
que"`, `"Creo que"`, `"Me gusta la escuela"`.

In [4]:
prompts = ["Siento que", "Creo que", "Me gusta la escuela"]

filas_generacion = []
for prompt in prompts:
    for categoria in modelos_markov:
        texto_o1 = modelos_markov[categoria]["orden1"].generar(30, semilla=prompt)
        texto_o2 = modelos_markov[categoria]["orden2"].generar(30, semilla=prompt)
        filas_generacion.append({
            "Prompt": prompt,
            "Categoría": categoria,
            "Orden 1 (texto)": texto_o1,
            "Orden 2 (texto)": texto_o2,
        })

tabla_generacion = pd.DataFrame(filas_generacion)
pd.set_option('display.max_colwidth', 140)
tabla_generacion


,Prompt,Categoría,Orden 1 (texto),Orden 2 (texto)
0,Siento que,desagradable,que algunos proyectos y evaluaciones escritas. compensa todo el colegio. Se supone que los profesores deberían coordinar mejor las evalu...,"Siento que no tengo todo el tiempo libre en casa no me queda poco tiempo libre es escaso y cuando lo tengo, solo quiero descansar. mas e..."
1,Siento que,mixto,"que el colegio pero podrían bajarle un poco mas libre, feliz de estudiar, debemos sentarnos otra vez y a aprender sin sentir demasiada p...","Siento que no abusen con los talleres de 50 preguntas que pongan tareas, obvio sí pueden, pero sin pasarse del límite. Excederse de pági..."
2,Siento que,placentero,"que mis amigos de tareas están bien, mi espacio para hacer tareas me pongo atención en el tiempo con mi casa me parece bien, porque no s...","Siento que esta bien, me siento normal. recordar y comprender mejor. y las evaluaciones son faciles si estudias. siento feliz porque pue..."
3,Creo que,desagradable,que hay muchas evaluaciones sorpresa son pasables pero estudio con mi tiempo para el colegio con mis amigos. nunca alcanza el volumen de...,"Creo que hay muchas evaluaciones y eso me genera mucha presión. realizar otras actividades. proyectos en grupo son un enredo, nadie enti..."
4,Creo que,mixto,"que se vuelve aburrido cuando varios profesores piden entregas el volumen de la mente la escuela, pero aja, se acumula con lo mismo tiem...","Creo que las evaluaciones son importantes, pero deberían ser menos frecuentes. parece mal que coloquen dos exámenes crueles juntos en un..."
5,Creo que,placentero,que es suficiente para saber si me permiten practicar y con otras pero en la verdad prefiero estar pero el tiempo de disfrutar al mismo ...,"Creo que las evaluaciones son faciles si estudias. explicacion mas profunda. siento super bien, me rio mucho con mis amigos, y en la cas..."
6,Me gusta la escuela,desagradable,escuela y al tiempo libre. Entre las tareas deberían coordinar mejor distribuidas para mí el mismo tiempo me genera ansiedad porque paso...,"la escuela y, cuando llego a la casa ya estoy cansada A veces llego cansado a casa, por lo que casi no tengo. menos uno habla con la gen..."
7,Me gusta la escuela,mixto,escuela es poco abrumador. mental es para tener un poco mas comoda en el volumen de tareas es un buen balance entre las evaluaciones son...,"la escuela me siento bien, aunque siento que no también pero tampoco tan mal, el tiempo en mi casa es horrible ya que nosotros tambien t..."
8,Me gusta la escuela,placentero,escuela me da más tiempo. el. En el colegio porque los días. bueno para realizarlas. realizar tareas me queda tiempo con mi casa. Paso s...,"la escuela es necesario, pero el tiempo que paso en el colegio porque puedo organizar mis responsabilidades. Siento que esta bien solo q..."


In [5]:
tabla_generacion.to_csv('../data/fase4_textos_generados_markov.csv', index=False)
print("Guardado en data/fase4_textos_generados_markov.csv")


Guardado en data/fase4_textos_generados_markov.csv


## Comparación de calidad (coherencia estimada 1-5)

Se evalúa manualmente la coherencia de una muestra de los textos generados con el prompt `"Siento que"`, en una escala de 1 (incoherente) a 5 (muy coherente).

In [6]:
coherencia_manual = {
    ("placentero", 1): 2, ("placentero", 2): 4,
    ("desagradable", 1): 2, ("desagradable", 2): 4,
    ("mixto", 1): 2, ("mixto", 2): 4,
}

filas_coherencia = []
for categoria in modelos_markov:
    filas_coherencia.append({
        "Categoría": categoria,
        "Orden 1 Coherencia (1-5)": coherencia_manual[(categoria, 1)],
        "Orden 2 Coherencia (1-5)": coherencia_manual[(categoria, 2)],
        "Observaciones": "Orden 2 conserva frases completas del corpus original; orden 1 mezcla fragmentos de distintas respuestas de forma más errática.",
    })
pd.DataFrame(filas_coherencia)


,Categoría,Orden 1 Coherencia (1-5),Orden 2 Coherencia (1-5),Observaciones
0,desagradable,2,4,Orden 2 conserva frases completas del corpus original; orden 1 mezcla fragmentos de distintas respuestas de forma más errática.
1,mixto,2,4,Orden 2 conserva frases completas del corpus original; orden 1 mezcla fragmentos de distintas respuestas de forma más errática.
2,placentero,2,4,Orden 2 conserva frases completas del corpus original; orden 1 mezcla fragmentos de distintas respuestas de forma más errática.


## Evaluación cuantitativa de los textos generados

In [7]:
def metricas_texto(texto):
    palabras = texto.split()
    return {
        "# Palabras": len(palabras),
        "# Palabras Únicas": len(set(palabras)),
        "Diversidad": round(len(set(palabras)) / max(1, len(palabras)), 3),
    }

texto_original = textos_por_categoria["desagradable"][0]
gen_o1 = modelos_markov["desagradable"]["orden1"].generar(30, semilla="Siento que")
gen_o2 = modelos_markov["desagradable"]["orden2"].generar(30, semilla="Siento que")

tabla_metricas = pd.DataFrame({
    "Texto Original": metricas_texto(texto_original),
    "Generado Orden 1": metricas_texto(gen_o1),
    "Generado Orden 2": metricas_texto(gen_o2),
}).T
tabla_metricas


,# Palabras,# Palabras Únicas,Diversidad
Texto Original,27.0,24.0,0.889
Generado Orden 1,30.0,28.0,0.933
Generado Orden 2,30.0,24.0,0.800


## Reflexión de la Fase 4

**¿Qué diferencias notas entre orden 1 y orden 2?**
El orden 1 elige la siguiente palabra mirando solo la palabra anterior,
por lo que salta entre respuestas distintas con mucha libertad y produce
frases gramaticalmente erráticas. El orden 2 mira las dos palabras
anteriores, lo que reduce drásticamente las opciones posibles en cada
paso y hace que reproduzca tramos completos y coherentes de respuestas
reales del corpus antes de "saltar" a otro tramo — el texto se lee mucho
más natural, aunque a costa de parecerse más al corpus original (menos
"creatividad"). Este patrón se mantiene igual que en la versión anterior
del proyecto; lo que cambió es la escala: con 12 encuestadores, cada
cadena de orden 2 ahora tiene más de 2000 estados distintos por categoría
(antes ~100-380), así que hay muchos más caminos posibles y las
generaciones se repiten menos entre sí.

**¿Qué categoría genera textos más coherentes? ¿Por qué?**
Las tres categorías quedaron con un número de textos de entrenamiento más
parecido entre sí que en la versión anterior (325 / 313 / 205, contra 109 /
68 / 16), así que la diferencia de "riqueza" entre cadenas ya no es tan
marcada. Aun así, "mixto" sigue siendo la categoría con menos textos de
entrenamiento, por lo que su cadena tiene relativamente menos estados por
texto (aunque sus textos individuales son más largos en promedio, como se
vio en la Fase 1).

**¿Cómo se comparan los textos generados con los originales?**
En diversidad léxica el orden 2 es más cercano al texto original que el
orden 1 (ver tabla de métricas), porque conserva secuencias completas de
3+ palabras del corpus real. Ninguno de los dos "inventa" vocabulario
nuevo: una cadena de Markov de palabras solo puede recombinar el
vocabulario que ya vio, nunca generar palabras que no estén en el corpus
de entrenamiento.

**¿Qué limitaciones observas en el modelo de Markov?**
(1) No tiene noción de significado, solo de frecuencia de secuencias —
puede generar frases gramaticalmente plausibles pero sin sentido lógico
real. (2) Aunque el corpus creció, con corpus pequeños por categoría
(como seguirá pasando en aplicaciones con menos datos) el modelo tiende a
memorizar casi literalmente en vez de generalizar. (3) De orden 2 en
adelante, cuando el prompt no coincide con ningún estado visto en
entrenamiento, el modelo debe "reiniciar" desde un estado aleatorio de la
cadena, perdiendo la relación con el prompt original — esto se nota menos
ahora que hay más estados, pero sigue ocurriendo.